###  Imports

In [1]:
import pickle
import faiss
import numpy as np

### Load Files

In [2]:
with open("data/metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

with open("data/vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

index = faiss.read_index("data/faiss_index.bin")

print("Files loaded successfully!")

Files loaded successfully!


### Stage 1 Retrieval

In [3]:
def retrieve_candidates(question, top_k=5):

    query_vector = vectorizer.transform(
        [question]
    ).toarray().astype("float32")

    distances, indices = index.search(
        query_vector,
        top_k
    )

    candidates = []

    for idx in indices[0]:

        candidates.append({

            "text": metadata[idx]["text"],

            "source": metadata[idx]["source"]

        })

    return candidates

### Reranking Function
Instead of a heavy CrossEncoder (which may cause installation problems), we'll simulate reranking using keyword overlap.

In [4]:
def rerank(question, candidates):

    question_words = set(question.lower().split())

    scored_results = []

    for item in candidates:

        text_words = set(item["text"].lower().split())

        score = len(

            question_words.intersection(
                text_words
            )

        )

        scored_results.append({

            "score": score,

            "text": item["text"],

            "source": item["source"]

        })

    scored_results.sort(

        key=lambda x: x["score"],

        reverse=True

    )

    return scored_results

### Multi-Stage Retrieval Pipeline

In [5]:
def multi_stage_search(question):

    candidates = retrieve_candidates(

        question,

        top_k=5

    )

    reranked = rerank(

        question,

        candidates

    )

    return reranked

### Test Query

In [6]:
results = multi_stage_search(

    "What monitoring tools are used?"

)

results

[{'score': 1,
  'text': 'Cloud Migration Guide\n\nAWS is the primary cloud platform.\n\nKubernetes is used for deployment.\n\nGitHub Actions manages CI/CD.\n\nPrometheus and Grafana are used for monitoring.',
  'source': 'cloud_migration_guide.txt'},
 {'score': 1,
  'text': 'Employee Handbook\n\nEmployees receive 20 annual leave days.\n\nRemote work is allowed three days per week.\n\nWorking hours are 9 AM to 6 PM.\n\nHealth insurance benefits are provided.',
  'source': 'employee_handbook.txt'},
 {'score': 0,
  'text': 'Company Policies\n\nPasswords must be changed every 90 days.\n\nVPN access is mandatory.\n\nMulti-factor authentication is required.\n\nConfidential data should not be shared externally.',
  'source': 'company_policies.txt'},
 {'score': 0,
  'text': 'EmployeeID     Name      Department  Experience\n0         101    Alice    Data Science           5\n1         102      Bob  ML Engineering           3\n2         103  Charlie              HR           8\n3         104    

### Show Top Result

In [7]:
best_result = results[0]

print("=" * 50)

print("Source:")
print(best_result["source"])

print("\nAnswer:")
print(best_result["text"])

print("\nScore:")
print(best_result["score"])

print("=" * 50)

Source:
cloud_migration_guide.txt

Answer:
Cloud Migration Guide

AWS is the primary cloud platform.

Kubernetes is used for deployment.

GitHub Actions manages CI/CD.

Prometheus and Grafana are used for monitoring.

Score:
1


### Compare Before vs After

In [8]:
original = retrieve_candidates(

    "What monitoring tools are used?",

    top_k=3

)

print("Before Reranking:\n")

for item in original:

    print(item["source"])

Before Reranking:

cloud_migration_guide.txt
employee_handbook.txt
company_policies.txt


In [9]:
reranked = multi_stage_search(

    "What monitoring tools are used?"

)

print("\nAfter Reranking:\n")

for item in reranked:

    print(item["source"])


After Reranking:

cloud_migration_guide.txt
employee_handbook.txt
company_policies.txt
employees.csv
products.csv


### Save Sample Results

In [10]:
import pandas as pd

df = pd.DataFrame(reranked)

df.to_csv(

    "outputs/reranking_results.csv",

    index=False

)

print("Results saved!")

Results saved!


### Verify Outputs

In [11]:
import os

print(os.listdir("outputs"))

['evaluation_metrics.csv', 'query_analytics.csv', 'reranking_results.csv']
